In [41]:
from langgraph.graph import StateGraph, START, END, add_messages
from typing import TypedDict, Annotated, List
from langgraph.types import Command
from langgraph.checkpoint.memory import MemorySaver
from langchain_groq import ChatGroq
from langchain_community.tools import TavilySearchResults
from langgraph.prebuilt import ToolNode
from langchain_core.messages import HumanMessage

memory = MemorySaver()

search_tool = TavilySearchResults(max_results=2)
tools = [search_tool]

llm = ChatGroq(model="llama-3.1-8b-instant")
llm_with_tools = llm.bind_tools(tools=tools)

class BasicState(TypedDict): 
    messages: Annotated[List, add_messages]

def model(state: BasicState): 
    return {
        "messages": [llm_with_tools.invoke(state["messages"])]
    }

def tools_router(state: BasicState): 
    last_message = state["messages"][-1]
    if(hasattr(last_message, "tool_calls") and 
    len(last_message.tool_calls) > 0):
        return "tools"
    else: 
        return END


graph = StateGraph(BasicState)
graph.add_node(model, "model")
graph.add_node("tools", ToolNode(tools=tools))

graph.set_entry_point("model")
graph.add_conditional_edges("model", tools_router)

graph.add_edge("tools", "model")

app = graph.compile(checkpointer=memory, interrupt_before=["tools"])

In [47]:
config = {
    "configurable":{
        'thread_id': 1
    }
}

events = app.stream({
    "messages": [HumanMessage(content="what is todays weather in chennai?")]
}, config=config, stream_mode="values")

for event in events:
    event["messages"][-1].pretty_print()

================================ Human Message =================================

what is todays weather in chennai?
================================== Ai Message ==================================
Tool Calls:
  tavily_search_results_json (zs9hrv3c6)
 Call ID: zs9hrv3c6
  Args:
    query: chennai weather today


In [48]:
snapshot = app.get_state(config)
print(snapshot)
print('next node:', snapshot.next)

StateSnapshot(values={'messages': [HumanMessage(content='what is current weather in chennai?', additional_kwargs={}, response_metadata={}, id='db464200-f930-49d6-a5bd-681135d1e253'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'qzrhqhyn7', 'function': {'arguments': '{"query":"current weather in chennai"}', 'name': 'tavily_search_results_json'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 286, 'total_tokens': 308, 'completion_time': 0.040009898, 'completion_tokens_details': None, 'prompt_time': 0.036239618, 'prompt_tokens_details': None, 'queue_time': 0.048067072, 'total_time': 0.076249516}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ee6ff-2a8a-7870-8ca2-b7046d4a0f45-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'current weather in che

In [ ]:
events = app.stream(Command(resume="yes"), config, stream_mode='values')

for event in events:
    event["messages"][-1].pretty_print()

================================== Ai Message ==================================
Tool Calls:
  tavily_search_results_json (zs9hrv3c6)
 Call ID: zs9hrv3c6
  Args:
    query: chennai weather today
================================= Tool Message =================================
Name: tavily_search_results_json

[{"title": "Chennai weather in June 2026 | Chennai 14 day weather", "url": "https://www.weather25.com/asia/india/tamil-nadu/chennai?page=month&month=June", "content": "Partly cloudy\nPartly cloudy\nSunny\nPartly cloudy\nPatchy rain possible\nPatchy rain possible\nThundery outbreaks possible\nThundery outbreaks possible\nThundery outbreaks possible\nPartly cloudy\nPatchy rain possible\nPatchy light drizzle\nPatchy rain possible\nPatchy rain possible\nPartly cloudy\nPatchy rain possible\nThundery outbreaks possible\nThundery outbreaks possible\nPartly cloudy\nCloudy\nThundery outbreaks possible\nLight rain shower\nPatchy rain possible\nPatchy rain possible\nCloudy\nLight rain shower\